# Create Genie Space Programmatically with Databricks SDK & Widgets

This notebook allows you to create a Genie space programmatically using the **Databricks SDK** and widgets for easy configuration. Perfect for automating Genie space creation across different environments!

- Databricks SDK Documentation: [https://docs.databricks.com/dev-tools/sdk-python.html](https://docs.databricks.com/dev-tools/sdk-python.html)
- Genie API Reference: [https://docs.databricks.com/api/workspace/genie/createspace](https://docs.databricks.com/api/workspace/genie/createspace)
- More about Genie: [https://docs.databricks.com/aws/en/genie/conversation-api](https://docs.databricks.com/aws/en/genie/conversation-api)

### Benefits of using the SDK:
- **Cleaner code** - No need to manage HTTP requests manually
- **Automatic authentication** - Uses your notebook's authentication context
- **Type safety** - Better IDE support and error checking
- **Built-in retries** - Handles transient errors automatically
- **Easy parameterization** - Use widgets for different environments

### Prerequisites:
Make sure you have the latest Databricks SDK installed (run the cell below if needed)


In [0]:
# Install or upgrade the Databricks SDK to the latest version
# Uncomment and run this cell if you need to install/upgrade the SDK

%pip install --upgrade databricks-sdk
dbutils.library.restartPython()  # Restart Python to use the new version


In [0]:
# Create Databricks Widgets for configuration
dbutils.widgets.text("space_title", "My New Genie Space", "1. Space Title")
dbutils.widgets.text("space_description", "", "2. Space Description (Optional)")
dbutils.widgets.text("warehouse_id", "", "3. Warehouse ID")
dbutils.widgets.text("table_identifiers", "", "4. Tables (Optional - comma separated)")
dbutils.widgets.dropdown("include_sample_instructions", "Yes", ["Yes", "No"], "5. Include Sample Instructions")

print("✓ Widgets created successfully!")
print("\nNote: The SDK automatically uses your notebook's authentication context.")
print("No need to provide workspace URL or credentials separately.")
print("\n💡 Tip: For tables, use format: catalog.schema.table")
print("   Example: main.sales.orders, main.sales.customers")


### Authentication with Databricks SDK

**Good News!** When using the Databricks SDK in a notebook, authentication is **automatic**. The SDK uses your notebook's execution context, so you don't need to:
- ❌ Manage Personal Access Tokens (PATs)
- ❌ Store secrets in secret scopes
- ❌ Handle authentication headers manually

The SDK automatically inherits your workspace credentials! 🎉

#### (Optional) For Advanced Use Cases

If you need to authenticate to a **different workspace** or use **custom credentials**, you can still do that:

```python
from databricks.sdk import WorkspaceClient

# Option 1: Use a specific PAT
w = WorkspaceClient(
    host="https://your-workspace.cloud.databricks.com",
    token=dbutils.secrets.get(scope="my-scope", key="my-key")
)

# Option 2: Use environment variables
# Set DATABRICKS_HOST and DATABRICKS_TOKEN in your environment
w = WorkspaceClient()
```

But for most use cases, just use `WorkspaceClient()` without any parameters!


### How to Find Your Warehouse ID

Your **Warehouse ID** is the unique identifier for your SQL Warehouse (compute resource).

#### Method 1: Using the Databricks UI

1. Go to **SQL Warehouses** in your Databricks workspace sidebar
2. Click on the warehouse you want to use
3. Look at the URL - the warehouse ID is at the end:
   - URL format: `https://<workspace>/sql/warehouses/<warehouse-id>`
   - Or check the **Connection Details** tab

#### Method 2: List Warehouses Programmatically


In [0]:
# List all SQL Warehouses and their IDs
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
warehouses = w.warehouses.list()

print("Available SQL Warehouses:")
print("-" * 80)
for wh in warehouses:
    print(f"Name: {wh.name}")
    print(f"ID: {wh.id}")
    print(f"State: {wh.state}")
    print("-" * 80)


In [0]:
from databricks.sdk import WorkspaceClient
import json

# Initialize the Databricks SDK client
# The SDK automatically uses the notebook's authentication context
w = WorkspaceClient()

# Get widget values
space_title = dbutils.widgets.get("space_title")
space_description = dbutils.widgets.get("space_description")
warehouse_id = dbutils.widgets.get("warehouse_id")
table_identifiers = dbutils.widgets.get("table_identifiers")
include_instructions = dbutils.widgets.get("include_sample_instructions")

# Validate required fields
if not space_title:
    raise ValueError("Space Title is required")
if not warehouse_id:
    raise ValueError("Warehouse ID is required")

print("✓ Configuration validated successfully")
print(f"✓ Space Title: {space_title}")
print(f"✓ Warehouse ID: {warehouse_id}")
if table_identifiers:
    tables_list = [t.strip() for t in table_identifiers.split(",") if t.strip()]
    print(f"✓ Tables: {len(tables_list)} table(s) specified")
    for table in tables_list:
        print(f"   - {table}")
else:
    print(f"ℹ️  No tables specified")
print(f"✓ SDK Client initialized")


### Build the Genie Space Configuration

Now we'll construct the serialized space configuration. The structure must match the API format:

```json
{
  "version": 1,
  "config": {
    "instructions": [...],
    "sample_questions": [...]
  },
  "data_sources": {
    "tables": [...]
  }
}
```

Key points:
- `version` is at the **root level** (not inside a nested object)
- `version` must be an **integer 1**, not a string
- The structure has three main keys: `version`, `config`, and `data_sources`


In [0]:
# Build the serialized space configuration
# Based on the actual API structure (version at root level, not inside genie_config)
serialized_space_config = {
    "version": 1,
    "config": {},
    "data_sources": {}
}

# Add sample instructions if requested
if include_instructions == "Yes":
    serialized_space_config["config"]["instructions"] = [
        "Always provide clear and concise answers",
        "When showing data, include relevant context",
        "Explain any assumptions made in your analysis"
    ]
    print("✓ Sample instructions included")
else:
    print("ℹ️  No sample instructions - using minimal configuration")

# Convert to JSON string (Genie API expects serialized_space as a JSON string)
serialized_space = json.dumps(serialized_space_config)

print("\n📋 Serialized Space Configuration:")
print(json.dumps(serialized_space_config, indent=2))
print("\n📝 Serialized as string:")
print(serialized_space)


### Create the Genie Space

Now let's use the SDK to create the new Genie space with our configuration.


In [0]:
print("🚀 Creating Genie Space using Databricks SDK...\n")

try:
    # Create the Genie Space using the SDK
    # The SDK handles authentication, retries, and error handling automatically
    # Note: The parameter is 'title' not 'display_name'
    created_space = w.genie.create_space(
        title=space_title,
        description=space_description if space_description else None,
        warehouse_id=warehouse_id,
        serialized_space=serialized_space
    )
    
    # Success! Display the results
    print("✅ Genie Space created successfully!\n")
    print("=" * 60)
    print(f"Space ID: {created_space.space_id}")
    print(f"Title: {created_space.title}")
    print(f"Warehouse ID: {created_space.warehouse_id}")
    if created_space.description:
        print(f"Description: {created_space.description}")
    if hasattr(created_space, 'created_timestamp'):
        print(f"Created At: {created_space.created_timestamp}")
    if hasattr(created_space, 'updated_timestamp'):
        print(f"Updated At: {created_space.updated_timestamp}")
    print("=" * 60)
    
    # Store the space_id for future use in workflow task values
    if created_space.space_id:
        try:
            dbutils.jobs.taskValues.set(key="genie_space_id", value=created_space.space_id)
            print(f"\n✓ Space ID saved to task values: {created_space.space_id}")
        except:
            # Task values might not be available if not running in a job
            print(f"\nℹ️  Space ID: {created_space.space_id} (task values not available in interactive mode)")
    
    # Display the full space object for reference
    print(f"\n📄 Full Space Object:")
    print(f"{created_space}")
    
except Exception as e:
    print(f"❌ Error creating Genie Space:")
    print(f"Error: {str(e)}")
    raise


### Advanced: Customize Data Sources and Sample Questions (Optional)

You can create a more advanced configuration with specific tables and sample questions:

```python
# Example with data sources and sample questions
advanced_config = {
    "version": 1,
    "config": {
        "instructions": ["Focus on sales metrics", "Compare period over period"],
        "sample_questions": [
            {
                "id": "q1",
                "question": ["Show total revenue by region"]
            },
            {
                "id": "q2",
                "question": ["What are the top 10 products by sales?"]
            }
        ]
    },
    "data_sources": {
        "tables": [
            {"identifier": "catalog.schema.sales_table"},
            {"identifier": "catalog.schema.products_table"}
        ]
    }
}

# Use this config instead of the simple one above
# serialized_space = json.dumps(advanced_config)
```


### Additional SDK Operations

The Databricks SDK also supports other Genie operations like listing, getting, updating, and deleting spaces.


In [0]:
# List all Genie spaces
all_spaces_response = w.genie.list_spaces()
all_spaces = all_spaces_response.spaces  # Access the list attribute

print("📋 All Genie Spaces in this workspace:")
print("-" * 80)
for space in all_spaces:
    print(f"Name: {space.title}")
    print(f"Space ID: {space.space_id}")
    print(f"Warehouse ID: {space.warehouse_id}")
    print("-" * 80)

### Clean Up Widgets (Optional)

Run this cell to remove all widgets when you're done.


In [0]:
# Uncomment to remove all widgets
# dbutils.widgets.removeAll()
# print("All widgets removed")
